In [9]:
from google.colab import files
import pandas as pd
import numpy as np
import re
import os

uploaded = files.upload()
filename = next(iter(uploaded))

if filename.lower().endswith((".xlsx", ".xls")):
    xls = pd.ExcelFile(filename)
    print("Available sheets:", xls.sheet_names)
    df = pd.read_excel(filename, sheet_name=xls.sheet_names[0])
elif filename.lower().endswith(".csv"):
    df = pd.read_csv(filename)
else:
    raise ValueError("Upload an Excel or CSV file.")

print("Original rows:", len(df))
print("Original columns:", df.columns.tolist())
display(df.head())

Saving 1-s2.0-S0273230026001364-mmc1_1285tox.xlsx to 1-s2.0-S0273230026001364-mmc1_1285tox (1).xlsx
Available sheets: ['Compiled Dataset']
Original rows: 1289
Original columns: ['ID', 'Reported CAS', 'Reported Chemical Name', 'Molecular Formula', 'SMILES', 'Tier 1 clusters', 'Tier 2 Clusters']


,ID,Reported CAS,Reported Chemical Name,Molecular Formula,SMILES,Tier 1 clusters,Tier 2 Clusters
0,1,102-79-4,N-Butyldiethanolamine,C8H19NO2,CCCCN(CCO)CCO,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines
1,2,1541-67-9,N-Lauryldiethanolamine,C16H35NO2,OCCN(CCO)CCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines
2,3,18312-57-7,"2,2-(Tridecylazanediyl)diethanol",C17H37NO2,OCCN(CCO)CCCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines
3,4,122-20-3,Triisopropanolamine,C9H21NO3,CC(CN(CC(O)C)CC(O)C)O,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines
4,5,111-42-2,Diethanolamine,C4H11NO2,OCCNCCO,"Aminoalcohol derivatives (1,2)",Alkyl secondary amines


In [10]:
!pip -q install rdkit openpyxl

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Standardized columns:", df.columns.tolist())

def find_column(columns, keywords):
    for col in columns:
        if any(keyword in col for keyword in keywords):
            return col
    return None

cas_col = find_column(df.columns, ["cas"])
smiles_col = find_column(df.columns, ["smiles", "structure"])
name_col = find_column(
    df.columns, ["chemical_name", "compound_name", "substance", "name"]
)

print("CAS column:", cas_col)
print("SMILES column:", smiles_col)
print("Name column:", name_col)

if smiles_col is None:
    raise ValueError(
        "Could not find the SMILES column. Rename the correct column to 'smiles'."
    )

Standardized columns: ['id', 'reported_cas', 'reported_chemical_name', 'molecular_formula', 'smiles', 'tier_1_clusters', 'tier_2_clusters']
CAS column: reported_cas
SMILES column: smiles
Name column: reported_chemical_name


In [11]:
clean = df.copy()
clean.insert(0, "original_row", range(2, len(clean) + 2))

for col in [cas_col, smiles_col, name_col]:
    if col is not None:
        clean[col] = clean[col].astype("string").str.strip()
        clean[col] = clean[col].replace("", pd.NA)

def normalize_cas(value):
    if pd.isna(value):
        return pd.NA
    value = re.sub(r"\s+", "", str(value))
    match = re.fullmatch(r"(\d{2,7})-(\d{2})-(\d)", value)
    return "-".join(match.groups()) if match else value

if cas_col:
    clean["cas_normalized"] = clean[cas_col].apply(normalize_cas)
    clean["cas_format_valid"] = clean["cas_normalized"].apply(
        lambda x: bool(re.fullmatch(r"\d{2,7}-\d{2}-\d", str(x)))
        if pd.notna(x) else False
    )
else:
    clean["cas_normalized"] = pd.NA
    clean["cas_format_valid"] = False

def validate_smiles(value):
    if pd.isna(value):
        return pd.Series([False, pd.NA, "Missing SMILES"])

    mol = Chem.MolFromSmiles(str(value))

    if mol is None:
        return pd.Series([False, pd.NA, "Invalid SMILES"])

    canonical = Chem.MolToSmiles(mol, canonical=True)
    return pd.Series([True, canonical, "Valid SMILES"])

clean[["smiles_valid", "canonical_smiles", "smiles_status"]] = (
    clean[smiles_col].apply(validate_smiles)
)

clean["structure_key"] = clean["canonical_smiles"]

print("Total records:", len(clean))
print("Valid SMILES:", int(clean["smiles_valid"].sum()))
print("Invalid/missing SMILES:", int((~clean["smiles_valid"]).sum()))
print("Missing CAS:", int(clean["cas_normalized"].isna().sum()))
print("CAS format issues:", int((~clean["cas_format_valid"]).sum()))

Total records: 1289
Valid SMILES: 1286
Invalid/missing SMILES: 3
Missing CAS: 13
CAS format issues: 68


[08:11:03] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:11:03] SMILES Parse Error: check for mistakes around position 71:
[08:11:03] (C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C
[08:11:03] ~~~~~~~~~~~~~~~~~~~~^
[08:11:03] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:11:03] SMILES Parse Error: check for mistakes around position 157:
[08:11:03] (C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8C
[08:11:03] ~~~~~~~~~~~~~~~~~~~~^
[08:11:03] SMILES Parse Error: Failed parsing SMILES 'CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5C

In [12]:
def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)

    return {
        "molecular_weight": Descriptors.MolWt(mol),
        "exact_molecular_weight": Descriptors.ExactMolWt(mol),
        "clogp": Descriptors.MolLogP(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "hbd": rdMolDescriptors.CalcNumHBD(mol),
        "hba": rdMolDescriptors.CalcNumHBA(mol),
        "rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol)
    }

valid_mask = clean["smiles_valid"].eq(True)

descriptor_df = clean.loc[valid_mask, "canonical_smiles"].apply(
    calculate_descriptors
).apply(pd.Series)

for col in descriptor_df.columns:
    clean[col] = np.nan
    clean.loc[valid_mask, col] = descriptor_df[col].values

display(clean.head())

,original_row,id,reported_cas,reported_chemical_name,molecular_formula,smiles,tier_1_clusters,tier_2_clusters,cas_normalized,cas_format_valid,...,molecular_weight,exact_molecular_weight,clogp,tpsa,hbd,hba,rotatable_bonds,heavy_atoms,ring_count,fraction_csp3
0,2,1,102-79-4,N-Butyldiethanolamine,C8H19NO2,CCCCN(CCO)CCO,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,102-79-4,True,...,161.245,161.141579,0.0731,43.70,2.0,3.0,7.0,11.0,0.0,1.0
1,3,2,1541-67-9,N-Lauryldiethanolamine,C16H35NO2,OCCN(CCO)CCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,1541-67-9,True,...,273.461,273.266779,3.1939,43.70,2.0,3.0,15.0,19.0,0.0,1.0
2,4,3,18312-57-7,"2,2-(Tridecylazanediyl)diethanol",C17H37NO2,OCCN(CCO)CCCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,18312-57-7,True,...,287.488,287.282429,3.5840,43.70,2.0,3.0,16.0,20.0,0.0,1.0
3,5,4,122-20-3,Triisopropanolamine,C9H21NO3,CC(CN(CC(O)C)CC(O)C)O,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,122-20-3,True,...,191.271,191.152144,-0.5692,63.93,3.0,4.0,6.0,13.0,0.0,1.0
4,6,5,111-42-2,Diethanolamine,C4H11NO2,OCCNCCO,"Aminoalcohol derivatives (1,2)",Alkyl secondary amines,111-42-2,True,...,105.137,105.078979,-1.4394,52.49,3.0,3.0,4.0,7.0,0.0,1.0


In [13]:
# Duplicate CAS numbers: preserve all records for audit
if cas_col:
    duplicate_cas = clean[
        clean["cas_normalized"].notna()
        & clean.duplicated("cas_normalized", keep=False)
    ].sort_values("cas_normalized")
else:
    duplicate_cas = clean.iloc[0:0].copy()

# Duplicate chemical structures
duplicate_structures = clean[
    clean["structure_key"].notna()
    & clean.duplicated("structure_key", keep=False)
].sort_values("structure_key")

# One representative row per valid structure
cleaned_structures = (
    clean[valid_mask]
    .drop_duplicates(subset="structure_key", keep="first")
    .copy()
)

# Invalid structures remain available for manual correction
needs_review = clean[~valid_mask].copy()

print("Original records:", len(clean))
print("Unique valid structures:", len(cleaned_structures))
print("Duplicate CAS rows:", len(duplicate_cas))
print("Duplicate structure rows:", len(duplicate_structures))
print("Records needing SMILES review:", len(needs_review))

Original records: 1289
Unique valid structures: 1286
Duplicate CAS rows: 4
Duplicate structure rows: 0
Records needing SMILES review: 3


In [14]:
output_file = "Borton_Cleaning_Report.xlsx"

summary = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Unique valid structures",
        "Valid SMILES",
        "Missing or invalid SMILES",
        "Missing CAS",
        "Invalid CAS format",
        "Rows with duplicate CAS",
        "Rows with duplicate structures",
        "Rows needing review"
    ],
    "Count": [
        len(clean),
        len(cleaned_structures),
        int(clean["smiles_valid"].sum()),
        int((~clean["smiles_valid"]).sum()),
        int(clean["cas_normalized"].isna().sum()),
        int((~clean["cas_format_valid"]).sum()),
        len(duplicate_cas),
        len(duplicate_structures),
        len(needs_review)
    ]
})

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    clean.to_excel(writer, sheet_name="All_Records_Audited", index=False)
    cleaned_structures.to_excel(writer, sheet_name="Cleaned_Structures", index=False)
    needs_review.to_excel(writer, sheet_name="Needs_Review", index=False)
    duplicate_cas.to_excel(writer, sheet_name="Duplicate_CAS", index=False)
    duplicate_structures.to_excel(writer, sheet_name="Duplicate_Structures", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)

print("Excel report created:", output_file)
display(summary)

files.download(output_file)

Excel report created: Borton_Cleaning_Report.xlsx


,Metric,Count
0,Original rows,1289
1,Unique valid structures,1286
2,Valid SMILES,1286
3,Missing or invalid SMILES,3
4,Missing CAS,13
5,Invalid CAS format,68
6,Rows with duplicate CAS,4
7,Rows with duplicate structures,0
8,Rows needing review,3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from google.colab import files
import pandas as pd
import numpy as np
import re
import os

uploaded = files.upload()
filename = next(iter(uploaded))

if filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(filename, sheet_name=0)
else:
    df = pd.read_csv(filename)

# Keep a backup of all original values
df_original = df.copy()

# Standardize column names
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Saving Borton_Cleaning_Report.xlsx to Borton_Cleaning_Report (1).xlsx
Rows: 1289
Columns: ['original_row', 'id', 'reported_cas', 'reported_chemical_name', 'molecular_formula', 'smiles', 'tier_1_clusters', 'tier_2_clusters', 'cas_normalized', 'cas_format_valid', 'smiles_valid', 'canonical_smiles', 'smiles_status', 'structure_key', 'molecular_weight', 'exact_molecular_weight', 'clogp', 'tpsa', 'hbd', 'hba', 'rotatable_bonds', 'heavy_atoms', 'ring_count']


,original_row,id,reported_cas,reported_chemical_name,molecular_formula,smiles,tier_1_clusters,tier_2_clusters,cas_normalized,cas_format_valid,...,structure_key,molecular_weight,exact_molecular_weight,clogp,tpsa,hbd,hba,rotatable_bonds,heavy_atoms,ring_count
0,2,1,102-79-4,N-Butyldiethanolamine,C8H19NO2,CCCCN(CCO)CCO,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,102-79-4,True,...,CCCCN(CCO)CCO,161.245,161.141579,0.0731,43.70,2.0,3.0,7.0,11.0,0.0
1,3,2,1541-67-9,N-Lauryldiethanolamine,C16H35NO2,OCCN(CCO)CCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,1541-67-9,True,...,CCCCCCCCCCCCN(CCO)CCO,273.461,273.266779,3.1939,43.70,2.0,3.0,15.0,19.0,0.0
2,4,3,18312-57-7,"2,2-(Tridecylazanediyl)diethanol",C17H37NO2,OCCN(CCO)CCCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,18312-57-7,True,...,CCCCCCCCCCCCCN(CCO)CCO,287.488,287.282429,3.5840,43.70,2.0,3.0,16.0,20.0,0.0
3,5,4,122-20-3,Triisopropanolamine,C9H21NO3,CC(CN(CC(O)C)CC(O)C)O,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,122-20-3,True,...,CC(O)CN(CC(C)O)CC(C)O,191.271,191.152144,-0.5692,63.93,3.0,4.0,6.0,13.0,0.0
4,6,5,111-42-2,Diethanolamine,C4H11NO2,OCCNCCO,"Aminoalcohol derivatives (1,2)",Alkyl secondary amines,111-42-2,True,...,OCCNCCO,105.137,105.078979,-1.4394,52.49,3.0,3.0,4.0,7.0,0.0


In [16]:
def normalize_cas(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip()

    if s.lower() in {"", "nan", "none", "n/a", "na", "unknown"}:
        return pd.NA

    # Remove common prefixes and whitespace
    s = re.sub(r"(?i)^cas\s*(?:no\.?|number)?\s*[:#-]?\s*", "", s)
    s = re.sub(r"\s+", "", s)
    s = s.strip("'\"")

    # Fix Excel numeric-looking strings such as 123456-78-9.0
    s = re.sub(r"\.0$", "", s)

    # Normalize separators if the digits clearly form CAS groups
    digits = re.sub(r"\D", "", s)
    if re.fullmatch(r"\d{5,10}", digits):
        s = f"{digits[:-3]}-{digits[-3:-1]}-{digits[-1]}"

    return s

def cas_checksum_valid(value):
    if pd.isna(value):
        return False

    m = re.fullmatch(r"(\d{2,7})-(\d{2})-(\d)", str(value))
    if not m:
        return False

    digits = m.group(1) + m.group(2)
    expected = sum(int(d) * i for i, d in enumerate(digits[::-1], start=1)) % 10

    return expected == int(m.group(3))

cas_col = next((c for c in df.columns if "cas" in c), None)
name_col = next(
    (c for c in df.columns
     if any(k in c for k in ["chemical_name", "compound_name", "substance", "name"])),
    None
)
smiles_col = next(
    (c for c in df.columns if "smiles" in c or "structure" in c),
    None
)

print("CAS column:", cas_col)
print("Name column:", name_col)
print("SMILES column:", smiles_col)

if cas_col is None:
    raise ValueError("CAS column not found. Rename the correct column to 'cas_number'.")

df["CAS_original"] = df[cas_col]
df["CAS_normalized"] = df[cas_col].apply(normalize_cas)
df["CAS_checksum_valid"] = df["CAS_normalized"].apply(cas_checksum_valid)

# Only automatically normalize entries that pass the checksum
valid_mask = df["CAS_checksum_valid"]
df.loc[valid_mask, cas_col] = df.loc[valid_mask, "CAS_normalized"]

df["CAS_review_status"] = np.select(
    [
        df["CAS_normalized"].isna(),
        df["CAS_checksum_valid"],
    ],
    [
        "Missing CAS — lookup needed",
        "CAS format and checksum valid — identity still needs verification",
    ],
    default="CAS needs manual review"
)

print(df["CAS_review_status"].value_counts(dropna=False))
display(df.loc[~df["CAS_checksum_valid"], [cas_col, "CAS_normalized", "CAS_review_status"]].head(20))

CAS column: reported_cas
Name column: reported_chemical_name
SMILES column: smiles
CAS_review_status
CAS format and checksum valid — identity still needs verification    1224
CAS needs manual review                                                52
Missing CAS — lookup needed                                            13
Name: count, dtype: int64


,reported_cas,CAS_normalized,CAS_review_status
6,145849-89-4; 193098-40-7,145849-89-4;193098-40-7,CAS needs manual review
7,192268-64-7; 195300-91-5,192268-64-7;195300-91-5,CAS needs manual review
14,25154-52-3; 104-40-5,25154-52-3;104-40-5,CAS needs manual review
19,123-94-4; 31566-31-1,123-94-4;31566-31-1,CAS needs manual review
51,14167-59-1; 14167-59-0,14167-59-1;14167-59-0,CAS needs manual review
89,111-30-08,1113-00-8,CAS needs manual review
155,26761-40-0; 68515-49-1,26761-40-0;68515-49-1,CAS needs manual review
175,58392-22-6; 131-57-7,58392-22-6;131-57-7,CAS needs manual review
207,465-29-2; 10373-78-1,465-29-2;10373-78-1,CAS needs manual review
210,38613-77-3; 119345-01-6,38613-77-3;119345-01-6,CAS needs manual review


In [17]:
!pip -q install requests rdkit openpyxl

import requests
import time
from urllib.parse import quote
from rdkit import Chem

if name_col is None:
    raise ValueError("Compound-name column not found. Rename it to 'compound_name'.")

def pubchem_lookup_by_name(name):
    """Return candidate PubChem CID, SMILES and CAS-like synonym, if found."""
    if pd.isna(name) or not str(name).strip():
        return {"CID_suggested": pd.NA, "SMILES_suggested": pd.NA,
                "CAS_suggested": pd.NA, "PubChem_status": "Missing compound name"}

    name = str(name).strip()
    base = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

    try:
        r = requests.get(
            f"{base}/compound/name/{quote(name, safe='')}/cids/JSON",
            timeout=20
        )
        if r.status_code != 200:
            return {"CID_suggested": pd.NA, "SMILES_suggested": pd.NA,
                    "CAS_suggested": pd.NA, "PubChem_status": f"No CID result ({r.status_code})"}

        cids = r.json().get("IdentifierList", {}).get("CID", [])
        if not cids:
            return {"CID_suggested": pd.NA, "SMILES_suggested": pd.NA,
                    "CAS_suggested": pd.NA, "PubChem_status": "No CID found"}

        # Use first result as a candidate, not as a confirmed identity
        cid = cids[0]

        p = requests.get(
            f"{base}/compound/cid/{cid}/property/CanonicalSMILES,IsomericSMILES/JSON",
            timeout=20
        )
        props = p.json().get("PropertyTable", {}).get("Properties", []) if p.ok else []
        smiles = pd.NA
        if props:
            smiles = props[0].get("ConnectivitySMILES") or props[0].get("CanonicalSMILES") or props[0].get("SMILES", pd.NA)

        s = requests.get(
            f"{base}/compound/cid/{cid}/synonyms/JSON",
            timeout=20
        )
        synonyms = []
        if s.ok:
            info = s.json().get("InformationList", {}).get("Information", [])
            if info:
                synonyms = info[0].get("Synonym", [])

        # Extract CAS-like synonyms and keep only checksum-valid candidates
        cas_candidates = []
        for synonym in synonyms:
            candidate = normalize_cas(synonym)
            if pd.notna(candidate) and cas_checksum_valid(candidate):
                if candidate not in cas_candidates:
                    cas_candidates.append(candidate)

        return {
            "CID_suggested": cid,
            "SMILES_suggested": smiles,
            "CAS_suggested": cas_candidates[0] if cas_candidates else pd.NA,
            "PubChem_status": "Candidate found — verify name/structure/CAS"
        }

    except Exception as e:
        return {"CID_suggested": pd.NA, "SMILES_suggested": pd.NA,
                "CAS_suggested": pd.NA, "PubChem_status": f"Lookup error: {type(e).__name__}"}

# Query records with missing CAS or invalid/missing SMILES
if smiles_col is None:
    df["SMILES_original"] = pd.NA
    df["SMILES_suggested"] = pd.NA
    df["SMILES_suggested_valid"] = False
else:
    df["SMILES_original"] = df[smiles_col]

needs_lookup = (
    df["CAS_normalized"].isna()
    | ~df["CAS_checksum_valid"]
)

if smiles_col:
    def smiles_is_valid(value):
        return pd.notna(value) and Chem.MolFromSmiles(str(value)) is not None

    needs_lookup |= ~df[smiles_col].apply(smiles_is_valid)

lookup_indices = df.index[needs_lookup].tolist()
print("Records to look up:", len(lookup_indices))

results = []
for idx in lookup_indices:
    result = pubchem_lookup_by_name(df.at[idx, name_col])
    result["row_index"] = idx
    results.append(result)
    time.sleep(0.25)  # be polite to the service

lookup_df = pd.DataFrame(results).set_index("row_index")

for col in ["CID_suggested", "SMILES_suggested", "CAS_suggested", "PubChem_status"]:
    df[col] = pd.NA
    if col in lookup_df.columns:
        df.loc[lookup_df.index, col] = lookup_df[col]

if smiles_col:
    df["SMILES_suggested_valid"] = df["SMILES_suggested"].apply(smiles_is_valid)

display(df.loc[needs_lookup, [name_col, cas_col, "CAS_suggested",
                              "SMILES_original", "SMILES_suggested",
                              "PubChem_status"]].head(30))

[08:16:13] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:16:13] SMILES Parse Error: check for mistakes around position 71:
[08:16:13] (C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C
[08:16:13] ~~~~~~~~~~~~~~~~~~~~^
[08:16:13] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:16:13] SMILES Parse Error: check for mistakes around position 157:
[08:16:13] (C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8C
[08:16:13] ~~~~~~~~~~~~~~~~~~~~^
[08:16:13] SMILES Parse Error: Failed parsing SMILES 'CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5C

Records to look up: 67


,reported_chemical_name,reported_cas,CAS_suggested,SMILES_original,SMILES_suggested,PubChem_status
6,"1,6-Hexanediamine,N,N'-bis(2,2,6,6-tetramethyl...",145849-89-4; 193098-40-7,<NA>,CC1=NC(N(CCCCCCN(C2CC(C)(C)N(C)C(C)(C)C2)C)C3C...,<NA>,No CID result (404)
7,"4-N-butyl-2-N,4-N-bis(2,2,6,6-tetramethylpiper...",192268-64-7; 195300-91-5,<NA>,CC1(C)NC(C)(C)CC(C1)N(C2=NC=NC(N(CCCC)C3CC(C)(...,<NA>,No CID result (404)
14,4-Nonylphenol,25154-52-3; 104-40-5,104-40-5,CCCCCCCCCC1=CC=C(O)C=C1,CCCCCCCCCC1=CC=C(C=C1)O,Candidate found — verify name/structure/CAS
19,"Glycerol Monostearate; 2,3-Dihydroxypropyl oct...",123-94-4; 31566-31-1,<NA>,O=C(OCC(O)CO)CCCCCCCCCCCCCCCCC,<NA>,No CID result (404)
51,n-Tetratriacontane,14167-59-1; 14167-59-0,14167-59-0,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC,Candidate found — verify name/structure/CAS
89,Glutaraldehyde,111-30-08,111-30-8,O=CCCCC=O,C(CC=O)CC=O,Candidate found — verify name/structure/CAS
155,Diisodecylphthalate,26761-40-0; 68515-49-1,<NA>,O=C(C1=CC=CC=C1C(OCCCCCCCC(C)C)=O)OCCCCCCCC(C)C,<NA>,No CID result (404)
175,2-Hydroxy-4-methoxybenzophenone,58392-22-6; 131-57-7,131-57-7,COC1=CC(O)=C(C(C2=CC=CC=C2)=O)C=C1,COC1=CC(=C(C=C1)C(=O)C2=CC=CC=C2)O,Candidate found — verify name/structure/CAS
207,"1-6N,71-6N,7-Trimethyl-bicyclo[2.2.1] heptane-...",465-29-2; 10373-78-1,<NA>,CC1(C2CCC1(C(C2=O)=O)C)C,<NA>,No CID result (404)
210,"tetrakis(2,4-di-tert-butylphenyl)-4,4'-bipheny...",38613-77-3; 119345-01-6,<NA>,CC(C)(C)C1=CC=C(C(C(C)(C)C)=C1)OP(OC2=CC=C(C(C...,<NA>,No CID result (404)


In [18]:
# Safe SMILES fill: only fill blank/invalid original SMILES with parseable suggestions
if smiles_col:
    for idx in df.index:
        original_valid = smiles_is_valid(df.at[idx, smiles_col])
        suggested = df.at[idx, "SMILES_suggested"]

        if not original_valid and smiles_is_valid(suggested):
            df.at[idx, smiles_col] = suggested
            df.at[idx, "SMILES_correction_status"] = (
                "Candidate SMILES inserted — verify identity"
            )
        else:
            df.at[idx, "SMILES_correction_status"] = (
                "Original SMILES retained" if original_valid
                else "Needs manual review"
            )

# CAS numbers are NOT automatically replaced with name-based candidates.
df["CAS_correction_status"] = np.where(
    df["CAS_checksum_valid"],
    "Format/checksum valid; verify chemical identity",
    np.where(
        df["CAS_suggested"].notna(),
        "CAS candidate available — verify before accepting",
        "Manual lookup required"
    )
)

output_file = "Borton_CAS_SMILES_Correction_Review.xlsx"

summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Missing CAS before lookup",
        "CAS checksum-valid after normalization",
        "CAS candidates suggested by PubChem",
        "SMILES candidates suggested by PubChem",
        "SMILES valid after candidate fill"
    ],
    "Count": [
        len(df),
        int(df["CAS_original"].isna().sum()),
        int(df["CAS_checksum_valid"].sum()),
        int(df["CAS_suggested"].notna().sum()),
        int(df["SMILES_suggested"].notna().sum()),
        int(df[smiles_col].apply(smiles_is_valid).sum()) if smiles_col else 0
    ]
})

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Correction_Review", index=False)
    df.loc[~df["CAS_checksum_valid"]].to_excel(
        writer, sheet_name="CAS_Needs_Review", index=False
    )
    if smiles_col:
        df.loc[~df[smiles_col].apply(smiles_is_valid)].to_excel(
            writer, sheet_name="SMILES_Needs_Review", index=False
        )
    summary.to_excel(writer, sheet_name="Summary", index=False)

print(summary.to_string(index=False))
files.download(output_file)

[08:17:36] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:17:36] SMILES Parse Error: check for mistakes around position 71:
[08:17:36] (C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C
[08:17:36] ~~~~~~~~~~~~~~~~~~~~^
[08:17:36] SMILES Parse Error: extra open parentheses while parsing: CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5CC(C)(C)N(C)C(C)(C)C5)=NC(N(CCCC)C6CC(C)(C)N(C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8CC(C)(C)N(C)C(C)(C)C8)=N
[08:17:36] SMILES Parse Error: check for mistakes around position 157:
[08:17:36] (C)C(C)(C)C6)=N4)CCN(CCCNC7=NC(N(CCCC)C8C
[08:17:36] ~~~~~~~~~~~~~~~~~~~~^
[08:17:36] SMILES Parse Error: Failed parsing SMILES 'CC(C1)(C)N(C)C(C)(C)CC1N(CCCC)C2=NC(N(CCCC)C3CC(C)(C)N(C)C(C)(C)C3)=NC(NCCCN(C4=NC(N(CCCC)C5C

                                Metric  Count
                         Total records   1289
             Missing CAS before lookup     13
CAS checksum-valid after normalization   1224
   CAS candidates suggested by PubChem     13
SMILES candidates suggested by PubChem     14
     SMILES valid after candidate fill   1287


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from google.colab import files
import pandas as pd
import numpy as np
import re
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

uploaded = files.upload()
input_file = next(iter(uploaded.keys()))

xls = pd.ExcelFile(input_file)
print("Input file:", input_file)
print("Sheets:", xls.sheet_names)

# Load the correction-review sheet if present
preferred = ["Correction_Review", "All_Records_Audited", "Cleaned_Structures"]
sheet = next((s for s in preferred if s in xls.sheet_names), xls.sheet_names[0])

df = pd.read_excel(input_file, sheet_name=sheet)
df.columns = df.columns.astype(str).str.strip()

print("Selected sheet:", sheet)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Saving Borton_CAS_SMILES_Correction_Review.xlsx to Borton_CAS_SMILES_Correction_Review (1).xlsx
Input file: Borton_CAS_SMILES_Correction_Review (1).xlsx
Sheets: ['Correction_Review', 'CAS_Needs_Review', 'SMILES_Needs_Review', 'Summary']
Selected sheet: Correction_Review
Rows: 1289
Columns: ['original_row', 'id', 'reported_cas', 'reported_chemical_name', 'molecular_formula', 'smiles', 'tier_1_clusters', 'tier_2_clusters', 'cas_normalized', 'cas_format_valid', 'smiles_valid', 'canonical_smiles', 'smiles_status', 'structure_key', 'molecular_weight', 'exact_molecular_weight', 'clogp', 'tpsa', 'hbd', 'hba', 'rotatable_bonds', 'heavy_atoms', 'ring_count', 'CAS_original', 'CAS_normalized', 'CAS_checksum_valid', 'CAS_review_status', 'SMILES_original', 'CID_suggested', 'SMILES_suggested', 'CAS_suggested', 'PubChem_status', 'SMILES_suggested_valid', 'SMILES_correction_status', 'CAS_correction_status']


,original_row,id,reported_cas,reported_chemical_name,molecular_formula,smiles,tier_1_clusters,tier_2_clusters,cas_normalized,cas_format_valid,...,CAS_checksum_valid,CAS_review_status,SMILES_original,CID_suggested,SMILES_suggested,CAS_suggested,PubChem_status,SMILES_suggested_valid,SMILES_correction_status,CAS_correction_status
0,2,1,102-79-4,N-Butyldiethanolamine,C8H19NO2,CCCCN(CCO)CCO,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,102-79-4,True,...,True,CAS format and checksum valid — identity still...,CCCCN(CCO)CCO,NaN,NaN,NaN,NaN,False,Original SMILES retained,Format/checksum valid; verify chemical identity
1,3,2,1541-67-9,N-Lauryldiethanolamine,C16H35NO2,OCCN(CCO)CCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,1541-67-9,True,...,True,CAS format and checksum valid — identity still...,OCCN(CCO)CCCCCCCCCCCC,NaN,NaN,NaN,NaN,False,Original SMILES retained,Format/checksum valid; verify chemical identity
2,4,3,18312-57-7,"2,2-(Tridecylazanediyl)diethanol",C17H37NO2,OCCN(CCO)CCCCCCCCCCCCC,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,18312-57-7,True,...,True,CAS format and checksum valid — identity still...,OCCN(CCO)CCCCCCCCCCCCC,NaN,NaN,NaN,NaN,False,Original SMILES retained,Format/checksum valid; verify chemical identity
3,5,4,122-20-3,Triisopropanolamine,C9H21NO3,CC(CN(CC(O)C)CC(O)C)O,"Aminoalcohol derivatives (1,2)",Alkyl tertiary amines,122-20-3,True,...,True,CAS format and checksum valid — identity still...,CC(CN(CC(O)C)CC(O)C)O,NaN,NaN,NaN,NaN,False,Original SMILES retained,Format/checksum valid; verify chemical identity
4,6,5,111-42-2,Diethanolamine,C4H11NO2,OCCNCCO,"Aminoalcohol derivatives (1,2)",Alkyl secondary amines,111-42-2,True,...,True,CAS format and checksum valid — identity still...,OCCNCCO,NaN,NaN,NaN,NaN,False,Original SMILES retained,Format/checksum valid; verify chemical identity


In [20]:
def split_cas(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "n/a", "na"}:
        return []

    # Extract CAS-like identifiers without silently changing the original
    return re.findall(r"(?<!\d)\d{2,7}-\d{2}-\d(?!\d)", text)


def normalize_cas(cas):
    if not cas:
        return None
    return str(cas).strip()


def cas_checksum_valid(cas):
    """CAS checksum check for one CAS number, not chemical identity."""
    if not cas or not re.fullmatch(r"\d{2,7}-\d{2}-\d", cas):
        return False

    digits = cas.replace("-", "")
    check_digit = int(digits[-1])
    body = digits[:-1]

    calculated = sum(
        int(digit) * weight
        for weight, digit in enumerate(reversed(body), start=1)
    ) % 10

    return calculated == check_digit


# Detect CAS column
cas_candidates = [
    c for c in df.columns
    if c.lower().replace("_", " ").strip() in {
        "reported cas", "cas", "cas number", "cas rn", "cas no",
        "reported cas number"
    }
]

if not cas_candidates:
    cas_candidates = [c for c in df.columns if "cas" in c.lower()]

if not cas_candidates:
    raise ValueError("No CAS column found. Check the printed column names.")

CAS_COL = "reported_cas" if "reported_cas" in df.columns else cas_candidates[0]
print("CAS column:", CAS_COL)

df["CAS_original_preserved"] = df[CAS_COL]
df["CAS_list"] = df[CAS_COL].apply(split_cas)
df["CAS_count_in_cell"] = df["CAS_list"].apply(len)
df["CAS_missing"] = df["CAS_count_in_cell"].eq(0)

# Row-level metrics: at least one CAS in the row passes checksum
df["CAS_checksum_valid_any"] = df["CAS_list"].apply(
    lambda xs: any(cas_checksum_valid(x) for x in xs)
)
df["CAS_checksum_invalid_all"] = df["CAS_list"].apply(
    lambda xs: len(xs) > 0 and not any(cas_checksum_valid(x) for x in xs)
)

# One CAS per row for a detailed audit
cas_audit = df[["CAS_original_preserved"]].copy()
cas_audit["CAS_list"] = df["CAS_list"]
cas_audit = cas_audit.explode("CAS_list").rename(
    columns={"CAS_list": "CAS_individual"}
)
cas_audit["CAS_individual"] = cas_audit["CAS_individual"].apply(normalize_cas)
cas_audit["CAS_checksum_valid"] = cas_audit["CAS_individual"].apply(
    cas_checksum_valid
)

print("Total records:", len(df))
print("Rows missing CAS:", int(df["CAS_missing"].sum()))
print("Rows containing multiple CAS numbers:", int((df["CAS_count_in_cell"] > 1).sum()))
print("Rows with at least one checksum-valid CAS:", int(df["CAS_checksum_valid_any"].sum()))
print("Individual CAS entries:", int(cas_audit["CAS_individual"].notna().sum()))
print("Individual CAS entries passing checksum:", int(cas_audit["CAS_checksum_valid"].sum()))

CAS column: reported_cas
Total records: 1289
Rows missing CAS: 14
Rows containing multiple CAS numbers: 51
Rows with at least one checksum-valid CAS: 1275
Individual CAS entries: 1352
Individual CAS entries passing checksum: 1332


In [21]:
# Find SMILES columns
def find_col(options, columns):
    lower_map = {c.lower().strip(): c for c in columns}
    for option in options:
        if option.lower() in lower_map:
            return lower_map[option.lower()]
    return None

SMILES_COL = find_col(
    ["smiles", "canonical_smiles", "original_smiles", "reported_smiles"],
    df.columns
)

SUGGESTED_SMILES_COL = find_col(
    ["SMILES_suggested", "suggested_smiles", "PubChem_SMILES",
     "pubchem_smiles_suggested"],
    df.columns
)

if SMILES_COL is None:
    raise ValueError("Could not identify the original SMILES column.")

print("Original SMILES column:", SMILES_COL)
print("Suggested SMILES column:", SUGGESTED_SMILES_COL)


def parse_smiles(smiles):
    if pd.isna(smiles) or not str(smiles).strip():
        return None
    try:
        return Chem.MolFromSmiles(str(smiles).strip())
    except Exception:
        return None


def canonicalize(smiles):
    mol = parse_smiles(smiles)
    return Chem.MolToSmiles(mol) if mol is not None else None


df["Original_SMILES_valid"] = df[SMILES_COL].apply(
    lambda x: parse_smiles(x) is not None
)
df["Original_SMILES_canonical"] = df[SMILES_COL].apply(canonicalize)

if SUGGESTED_SMILES_COL:
    df["Suggested_SMILES_valid"] = df[SUGGESTED_SMILES_COL].apply(
        lambda x: parse_smiles(x) is not None
    )
    df["Suggested_SMILES_canonical"] = df[SUGGESTED_SMILES_COL].apply(
        canonicalize
    )
else:
    df["Suggested_SMILES_valid"] = False
    df["Suggested_SMILES_canonical"] = None

# A row is structurally parseable if either source parses.
# The suggested structure remains separate pending identity verification.
df["Any_SMILES_parseable"] = (
    df["Original_SMILES_valid"] | df["Suggested_SMILES_valid"]
)

df["SMILES_review_status"] = np.select(
    [
        df["Original_SMILES_valid"],
        (~df["Original_SMILES_valid"]) & df["Suggested_SMILES_valid"]
    ],
    [
        "Original SMILES parses",
        "Suggested SMILES parses - identity needs verification"
    ],
    default="No parseable SMILES - manual review"
)

print("Total records:", len(df))
print("Original SMILES parseable:", int(df["Original_SMILES_valid"].sum()))
print("Suggested SMILES parseable:", int(df["Suggested_SMILES_valid"].sum()))
print("Parseable using original or suggestion:", int(df["Any_SMILES_parseable"].sum()))
print("Needs SMILES review:", int((~df["Any_SMILES_parseable"]).sum()))

display(
    df.loc[
        ~df["Any_SMILES_parseable"],
        [CAS_COL, SMILES_COL, "SMILES_review_status"]
    ].head(30)
)

Original SMILES column: smiles
Suggested SMILES column: SMILES_suggested


[08:31:34] SMILES Parse Error: syntax error while parsing: [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C
[08:31:34] SMILES Parse Error: check for mistakes around position 2:
[08:31:34] [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC
[08:31:34] ~^
[08:31:34] SMILES Parse Error: Failed parsing SMILES '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C' for input: '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C'
[08:31:34] Explicit valence for atom # 1 N, 5, is greater than permitted


Total records: 1289
Original SMILES parseable: 1287
Suggested SMILES parseable: 14
Parseable using original or suggestion: 1287
Needs SMILES review: 2


[08:31:34] SMILES Parse Error: syntax error while parsing: [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C
[08:31:34] SMILES Parse Error: check for mistakes around position 2:
[08:31:34] [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC
[08:31:34] ~^
[08:31:34] SMILES Parse Error: Failed parsing SMILES '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C' for input: '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C'
[08:31:34] Explicit valence for atom # 1 N, 5, is greater than permitted


,reported_cas,smiles,SMILES_review_status
816,90751-07-8,[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CC...,No parseable SMILES - manual review
1287,NaN,[O-][NH+]([C@@H]([C@@H]([C@@H]1CCCC2)C)[C@@H](...,No parseable SMILES - manual review


In [22]:
df["SMILES_for_analysis"] = df[SMILES_COL].astype("string")
df["SMILES_source_for_analysis"] = np.where(
    df["Original_SMILES_valid"], "Original",
    np.where(df["Suggested_SMILES_valid"], "PubChem suggestion - unverified",
             "Missing/invalid")
)

use_suggestion = (
    (~df["Original_SMILES_valid"]) &
    df["Suggested_SMILES_valid"]
)

df.loc[use_suggestion, "SMILES_for_analysis"] = (
    df.loc[use_suggestion, SUGGESTED_SMILES_COL]
    if SUGGESTED_SMILES_COL else df.loc[use_suggestion, "SMILES_for_analysis"]
)

df["Canonical_SMILES_for_analysis"] = df["SMILES_for_analysis"].apply(canonicalize)

# Descriptor calculation only for parseable structures
def calculate_descriptors(smiles):
    mol = parse_smiles(smiles)
    if mol is None:
        return pd.Series({
            "MolWt": np.nan, "ExactMolWt": np.nan, "cLogP": np.nan,
            "TPSA": np.nan, "HBD": np.nan, "HBA": np.nan,
            "RotatableBonds": np.nan, "HeavyAtoms": np.nan,
            "RingCount": np.nan, "FractionCSP3": np.nan
        })

    return pd.Series({
        "MolWt": Descriptors.MolWt(mol),
        "ExactMolWt": Descriptors.ExactMolWt(mol),
        "cLogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "RotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol)
    })

descriptor_df = df["SMILES_for_analysis"].apply(calculate_descriptors)
df = pd.concat([df, descriptor_df], axis=1)

[08:33:10] SMILES Parse Error: syntax error while parsing: [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C
[08:33:10] SMILES Parse Error: check for mistakes around position 2:
[08:33:10] [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC
[08:33:10] ~^
[08:33:10] SMILES Parse Error: Failed parsing SMILES '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C' for input: '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C'
[08:33:11] Explicit valence for atom # 1 N, 5, is greater than permitted
[08:33:15] SMILES Parse Error: syntax error while parsing: [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(C)C)CC(C)(N4)C
[08:33:15] SMILES Parse Error: check for mistakes around position 2:
[08:33:15] [[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC
[08:33:15] ~^
[08:33:15] SMILES Parse Error: Failed parsing SMILES '[[*]]N(C(CC1(C)C)CC(C)(N1)C)CCCCCCN(C2=NC(N3CCOCC3)=NC([[*]])=N2)C(CC4(

In [23]:
summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Missing CAS before lookup",
        "Rows with at least one checksum-valid CAS",
        "Individual CAS entries extracted",
        "Individual CAS entries passing checksum",
        "Rows containing multiple CAS numbers",
        "Original SMILES parseable",
        "Suggested SMILES parseable",
        "SMILES parseable using original or suggestion",
        "Rows requiring SMILES review",
        "Rows using a suggested SMILES for analysis",
        "Rows with descriptors calculated"
    ],
    "Count": [
        len(df),
        int(df["CAS_missing"].sum()),
        int(df["CAS_checksum_valid_any"].sum()),
        int(cas_audit["CAS_individual"].notna().sum()),
        int(cas_audit["CAS_checksum_valid"].sum()),
        int((df["CAS_count_in_cell"] > 1).sum()),
        int(df["Original_SMILES_valid"].sum()),
        int(df["Suggested_SMILES_valid"].sum()),
        int(df["Any_SMILES_parseable"].sum()),
        int((~df["Any_SMILES_parseable"]).sum()),
        int(use_suggestion.sum()),
        int(df["Canonical_SMILES_for_analysis"].notna().sum())
    ]
})

display(summary)

,Metric,Count
0,Total records,1289
1,Missing CAS before lookup,14
2,Rows with at least one checksum-valid CAS,1275
3,Individual CAS entries extracted,1352
4,Individual CAS entries passing checksum,1332
5,Rows containing multiple CAS numbers,51
6,Original SMILES parseable,1287
7,Suggested SMILES parseable,14
8,SMILES parseable using original or suggestion,1287
9,Rows requiring SMILES review,2


In [24]:
output_file = "Borton_Corrected_Audit.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Corrected_Records", index=False)
    summary.to_excel(writer, sheet_name="Summary", index=False)
    cas_audit.to_excel(writer, sheet_name="Individual_CAS_Audit", index=False)

    df.loc[df["CAS_missing"]].to_excel(
        writer, sheet_name="Missing_CAS", index=False
    )
    df.loc[df["CAS_count_in_cell"] > 1].to_excel(
        writer, sheet_name="Multiple_CAS_Review", index=False
    )
    df.loc[~df["Any_SMILES_parseable"]].to_excel(
        writer, sheet_name="SMILES_Needs_Review", index=False
    )
    df.loc[use_suggestion].to_excel(
        writer, sheet_name="Suggested_SMILES_Review", index=False
    )

print("Saved:", output_file)

from google.colab import files
files.download(output_file)

Saved: Borton_Corrected_Audit.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>